# Data Cleaning 04 -- IBES Recommendations

## Input
`Data/Data_Collection/Initial/04_LSEG_IBES/ibes_recommendations.parquet` (4,257,974 rows across 51,457 IBES tickers)

## Purpose
Cleans monthly stock-level analyst recommendation data from WRDS IBES (recddet). The raw file contains the entire IBES US coverage universe. This notebook filters to the top-100 S&P 500 universe, then validates the recommendation scale, NaN patterns, and structural consistency.

## Stage 0: Load, Filter to Universe, Inspect
- Raw file filtered from 51,457 tickers to the 231 tickers mapping to the 227 master PERMNOs using `ibes_permno_link_clean.parquet`
- Trimmed to 2004-01-01 onwards
- Reduced from 4,257,974 to 49,110 rows
- Basic shape, date range, ticker counts, and per-ticker coverage statistics reported

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts with flags for columns above 30%
- Per-row NaN distribution

## Stage 2: Value Range & Quality Checks
- **Recommendation scale:** verifies `rec_mean` and `rec_median` fall within [1, 5] (IBES scale: 1=Strong Buy, 5=Sell)
- **Analyst count:** distribution of `rec_numest`, fraction of single-analyst observations
- **Buy/Hold/Sell percentages:** range checks on `rec_buypct`, `rec_holdpct`, `rec_sellpct` and verification that they sum to ~100 (or ~1.0 depending on scale)
- **Upgrade/downgrade counts:** range, mean, and fraction of zero values for `rec_upgrade` and `rec_downgrade`
- **Revision:** distribution and percentile analysis of `rec_revision`
- **Dispersion:** distribution and percentile analysis of `rec_dispersion`
- **NaN pattern -- dispersion:** confirms `rec_dispersion` NaN occurs exclusively when `rec_numrec = 1`
- **NaN pattern -- revision:** confirms `rec_revision` NaN occurs exclusively in the first month per ticker
- **Coverage per ticker:** months of data per ticker, identifies tickers with fewer than 24 months
- **Duplicate (ticker, date) check**
- **Date frequency:** verifies dates are month-end

## Stage 4: Clean & Save

### Filtered to Universe (Stage 0)
Reduced from 4,257,974 rows (51,457 tickers) to 49,110 rows (231 tickers mapping to 227 PERMNOs), trimmed to 2004+.

### No Columns Dropped
All 13 factor columns retained. Winsorisation and z-standardisation will be applied cross-sectionally per date in the merge pipeline.

### No Winsorisation Applied Here
Value ranges are well-behaved for top-100 S&P 500 stocks: `rec_mean` and `rec_median` within [1, 5], `rec_revision` ranges -2.0 to +2.0 (extreme but plausible on a 5-point scale), `rec_dispersion` ranges 0 to 2.1. No outliers requiring treatment at this stage.

### Structural NaN Left as NaN (493 Total, 0.08% of Cells)
- `rec_dispersion` (377 NaN): 100% occur where `rec_numrec = 1`. With a single analyst, dispersion is undefined. Forward-filling would carry forward a multi-analyst dispersion into a single-analyst month, falsely implying disagreement.
- `rec_revision` (29 NaN): 100% are the first observation per ticker, no prior month to compute a change from.
- `rec_revision_3m` (87 NaN): first 3 observations per ticker, no 3-month-ago value to compare against.

These will be skipped during cross-sectional aggregation in the merge pipeline.

### Short-Coverage Tickers
3 tickers with fewer than 24 months of coverage (FBS, FLT, ONE) -- same stocks as in price targets, briefly in the top-100 universe then exited. Kept.

## Output
`Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_recommendations_clean.parquet` -- 13 factor columns (all retained), 49,110 rows

In [1]:
# %% [markdown]
# # Data Cleaning: ibes_recommendations.parquet
#
# Source: Data/Data_Collection/Initial/04_LSEG_IBES/ibes_recommendations.parquet
# Output: Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_recommendations_clean.parquet
#
# Monthly stock-level analyst recommendation data from WRDS IBES (recddet).
# Like price targets, the raw file contains ALL IBES tickers. We filter to
# our universe first using the clean link table.
#
# Expected factors (from collection code):
#   - rec_mean: mean recommendation (1=Strong Buy, 5=Sell)
#   - rec_median: median recommendation
#   - rec_numest: number of analysts
#   - rec_buypct / rec_holdpct / rec_sellpct: % of analysts in each bucket
#   - rec_upgrade / rec_downgrade: count of upgrades/downgrades
#   - rec_revision: change in mean recommendation
#   - rec_dispersion: std/mean of recommendations

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH  = Path('../../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_recommendations.parquet')
LINK_PATH = Path('../../../Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_permno_link_clean.parquet')
OUT_DIR   = Path('../../../Data/Data_Collection/Cleaned/04_LSEG_IBES')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD, FILTER TO UNIVERSE, INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD, FILTER TO UNIVERSE, INSPECT")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])
link = pd.read_parquet(LINK_PATH)

print(f"\n  Raw shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Raw tickers: {df['ticker'].nunique():,}")

# ── Filter to universe tickers ───────────────────────────────────────────────
valid_tickers = set(link['ticker'].unique())
n_before = len(df)
df = df[df['ticker'].isin(valid_tickers)].reset_index(drop=True)
print(f"\n  Filtered to universe tickers: {n_before:,} → {len(df):,} rows")
print(f"  Tickers retained: {df['ticker'].nunique()}")

# ── Trim to 2004-01-01 ──────────────────────────────────────────────────────
n_before = len(df)
df = df[df['date'] >= '2004-01-01'].reset_index(drop=True)
print(f"  Trimmed to 2004+: {n_before:,} → {len(df):,} rows")

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique()}")
print(f"  Unique tickers: {df['ticker'].nunique()}")
print(f"  Rows per ticker (mean): {df.groupby('ticker').size().mean():.1f}")
print(f"  Rows per ticker (median): {df.groupby('ticker').size().median():.0f}")

factor_cols = [c for c in df.columns if c not in ['ticker', 'date']]

print(f"\nColumns and dtypes ({len(factor_cols)} factors):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<25s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (10 rows) ---")
print(df.head(10).to_string(index=False))

print(f"\n--- Tail (10 rows) ---")
print(df.tail(10).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN ───────────────────────────────────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Column':<25s} {'NaN %':>8s}  {'Count':>8s}")
print("  " + "-" * 45)
for col, pct in col_nan_sorted.items():
    count = int(col_nan[col])
    flag = " ← DROP" if pct >= 30 else ""
    print(f"  {col:<25s} {pct:>7.2f}%  {count:>8,d}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():>8,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-3 NaN: {((row_nan >= 1) & (row_nan <= 3)).sum():>8,d}")
print(f"  Rows with 4-6 NaN: {((row_nan > 3) & (row_nan <= 6)).sum():>8,d}")
print(f"  Rows with >6 NaN: {(row_nan > 6).sum():>8,d}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: VALUE RANGE & QUALITY CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: VALUE RANGE & QUALITY CHECKS")
print("=" * 90)

# ── 2a. Recommendation scale ────────────────────────────────────────────────
print(f"\n--- Recommendation scale ---")
# IBES uses 1=Strong Buy, 2=Buy, 3=Hold, 4=Underperform, 5=Sell
for col in ['rec_mean', 'rec_median']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    print(f"  {col}:")
    print(f"    range: {vals.min():.4f} – {vals.max():.4f}")
    print(f"    mean: {vals.mean():.4f}, median: {vals.median():.4f}")
    n_outside = ((vals < 1) | (vals > 5)).sum()
    if n_outside > 0:
        print(f"    ⚠ {n_outside} values outside [1, 5]")
    else:
        print(f"    ✓ All values in [1, 5]")

# ── 2b. Analyst count ────────────────────────────────────────────────────────
if 'rec_numest' in df.columns:
    vals = df['rec_numest'].dropna()
    print(f"\n--- Analyst count (rec_numest) ---")
    print(f"  range: {vals.min():.0f} – {vals.max():.0f}")
    print(f"  mean: {vals.mean():.1f}, median: {vals.median():.0f}")
    print(f"  Stocks with 1 analyst: {(vals == 1).sum():,} ({(vals == 1).mean()*100:.1f}%)")

# ── 2c. Buy/Hold/Sell percentages ───────────────────────────────────────────
print(f"\n--- Buy/Hold/Sell percentages ---")
pct_cols = ['rec_buypct', 'rec_holdpct', 'rec_sellpct']
pct_cols_present = [c for c in pct_cols if c in df.columns]

for col in pct_cols_present:
    vals = df[col].dropna()
    print(f"  {col:<20s} range: {vals.min():.4f} – {vals.max():.4f}  "
          f"mean: {vals.mean():.4f}")

# Check if they sum to ~1.0 (or ~100)
if len(pct_cols_present) == 3:
    pct_sum = df[pct_cols_present].sum(axis=1).dropna()
    print(f"\n  Sum of buy+hold+sell:")
    print(f"    mean: {pct_sum.mean():.4f}, min: {pct_sum.min():.4f}, max: {pct_sum.max():.4f}")
    if pct_sum.mean() > 50:
        print(f"    → Percentages appear to be 0-100 scale")
    elif pct_sum.mean() < 2:
        print(f"    → Percentages appear to be 0-1 (decimal) scale")

# ── 2d. Upgrade/downgrade counts ────────────────────────────────────────────
print(f"\n--- Upgrade/downgrade counts ---")
for col in ['rec_upgrade', 'rec_downgrade']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    print(f"  {col}:")
    print(f"    range: {vals.min():.0f} – {vals.max():.0f}")
    print(f"    mean: {vals.mean():.2f}, median: {vals.median():.0f}")
    print(f"    % zero: {(vals == 0).mean()*100:.1f}%")

# ── 2e. Revision ────────────────────────────────────────────────────────────
for col in ['rec_revision']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    print(f"\n--- rec_revision (change in mean recommendation) ---")
    print(f"  range: {vals.min():.4f} – {vals.max():.4f}")
    print(f"  mean: {vals.mean():.4f}, median: {vals.median():.4f}")
    pctiles = vals.quantile([0.01, 0.05, 0.95, 0.99])
    print(f"  1st: {pctiles[0.01]:.4f}  5th: {pctiles[0.05]:.4f}  "
          f"95th: {pctiles[0.95]:.4f}  99th: {pctiles[0.99]:.4f}")

# ── 2f. Dispersion ──────────────────────────────────────────────────────────
if 'rec_dispersion' in df.columns:
    vals = df['rec_dispersion'].dropna()
    print(f"\n--- rec_dispersion ---")
    print(f"  range: {vals.min():.4f} – {vals.max():.4f}")
    print(f"  mean: {vals.mean():.4f}, median: {vals.median():.4f}")
    pctiles = vals.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
    print(f"  1st: {pctiles[0.01]:.4f}  25th: {pctiles[0.25]:.4f}  "
          f"median: {pctiles[0.50]:.4f}  75th: {pctiles[0.75]:.4f}  "
          f"99th: {pctiles[0.99]:.4f}")

# ── 2g. NaN pattern: dispersion when numest = 1 ─────────────────────────────
print(f"\n--- NaN pattern: rec_dispersion when numest = 1 ---")
if all(c in df.columns for c in ['rec_dispersion', 'rec_numest']):
    single_analyst = df['rec_numest'] == 1
    disp_nan = df['rec_dispersion'].isna()
    both = (single_analyst & disp_nan).sum()
    disp_nan_total = disp_nan.sum()
    print(f"  rec_dispersion NaN total: {disp_nan_total}")
    if disp_nan_total > 0:
        print(f"  Of those, where rec_numest = 1: {both} ({both/disp_nan_total*100:.1f}%)")

# ── 2h. NaN pattern: revision in first month per ticker ─────────────────────
print(f"\n--- NaN pattern: rec_revision in first month per ticker ---")
if 'rec_revision' in df.columns:
    rev_nan = df['rec_revision'].isna()
    first_month = df.groupby('ticker')['date'].transform('min') == df['date']
    both = (first_month & rev_nan).sum()
    rev_nan_total = rev_nan.sum()
    print(f"  rec_revision NaN total: {rev_nan_total}")
    if rev_nan_total > 0:
        print(f"  Of those, in first month per ticker: {both} ({both/rev_nan_total*100:.1f}%)")

# ── 2i. Coverage per ticker ─────────────────────────────────────────────────
print(f"\n--- Coverage per ticker ---")
ticker_coverage = df.groupby('ticker').agg(
    n_months=('date', 'nunique'),
    first_date=('date', 'min'),
    last_date=('date', 'max')
)
print(f"  Months per ticker: mean={ticker_coverage['n_months'].mean():.0f}, "
      f"median={ticker_coverage['n_months'].median():.0f}, "
      f"min={ticker_coverage['n_months'].min()}, "
      f"max={ticker_coverage['n_months'].max()}")

short_coverage = ticker_coverage[ticker_coverage['n_months'] < 24]
if len(short_coverage) > 0:
    print(f"\n  Tickers with <24 months: {len(short_coverage)}")
    for ticker, row in short_coverage.iterrows():
        print(f"    {ticker:<10s} {row['n_months']:>3d} months  "
              f"({row['first_date'].date()} → {row['last_date'].date()})")

# ── 2j. Duplicate (ticker, date) ────────────────────────────────────────────
print(f"\n--- Duplicate (ticker, date) ---")
n_dupes = df.duplicated(subset=['ticker', 'date']).sum()
if n_dupes == 0:
    print(f"  ✓ No duplicates")
else:
    print(f"  ⚠ {n_dupes} duplicates")

# ── 2k. Date frequency ──────────────────────────────────────────────────────
print(f"\n--- Date frequency ---")
is_month_end = df['date'].dt.is_month_end
print(f"  Dates that are month-end: {is_month_end.sum():,} ({is_month_end.mean()*100:.1f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: SUMMARY — DECISIONS NEEDED")
print("=" * 90)

print(f"""
Review the output above:

1. COLUMNS:
   - Any column with ≥30% NaN → drop
   - All others → keep (winsorisation and z-standardisation in merge pipeline)

2. NaN:
   - Structural NaN (dispersion with 1 analyst, revision in first month)
     → leave as NaN
   - Any unexpected NaN patterns → investigate

3. VALUE RANGES:
   - rec_mean/rec_median should be in [1, 5]
   - buy+hold+sell should sum to ~1.0 or ~100
   - Upgrade/downgrade counts should be non-negative integers

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD, FILTER TO UNIVERSE, INSPECT

  Raw shape: 4,257,974 rows × 15 columns
  Raw tickers: 51,457

  Filtered to universe tickers: 4,257,974 → 51,478 rows
  Tickers retained: 230
  Trimmed to 2004+: 51,478 → 49,110 rows

  Shape: 49,110 rows × 15 columns
  Date range: 2004-01-31 → 2024-12-31
  Unique dates: 252
  Unique tickers: 230
  Rows per ticker (mean): 213.5
  Rows per ticker (median): 252

Columns and dtypes (13 factors):
    1. rec_mean                  Float64        
    2. rec_median                Float64        
    3. rec_numrec                Int64          
    4. rec_buy_pct               Float64        
    5. rec_sell_pct              Float64        
    6. rec_buy_sell_spread       Float64        
    7. rec_dispersion            Float64        
    8. rec_revision              Float64        
    9. rec_revision_3m           Float64        
   10. rec_upgrades              float64        
   11. rec_downgrades            float64        
   12. rec_changes 

In [2]:
print((df[df['rec_numrec'] == 1]['rec_dispersion'].isna()).sum())

377


In [3]:
# %% [markdown]
# ## Stage 4: Clean & Save
#
# **Filtered to universe (Stage 0):**
# The raw file contains 4,257,974 rows across 51,457 IBES tickers. Filtered
# to the 231 tickers mapping to our 227 top-100 S&P 500 PERMNOs using the
# clean link table, then trimmed to 2004-01-31 onwards. This reduced the
# data from 4.26M to 49,110 rows.
#
# **No columns dropped.** All 13 factor columns retained. Winsorisation and
# z-standardisation will be applied cross-sectionally per date in the merge
# pipeline.
#
# **No winsorisation applied here.** Value ranges are well-behaved for
# top-100 S&P 500 stocks: `rec_mean` and `rec_median` are within [1, 5],
# `rec_revision` ranges −2.0 to +2.0 (extreme but plausible on a 5-point
# scale), `rec_dispersion` ranges 0 to 2.1. No outliers requiring treatment.
#
# **Structural NaN left as NaN (493 total, 0.08% of cells):**
# - `rec_dispersion` (377 NaN): 100% occur where `rec_numrec = 1`. With a
#   single analyst, dispersion (std/mean) is undefined. Forward-filling would
#   carry forward a multi-analyst dispersion into a single-analyst month,
#   falsely implying disagreement that doesn't exist.
# - `rec_revision` (29 NaN): 100% are the first observation per ticker — no
#   prior month to compute a change from. Cannot forward-fill (no prior row).
# - `rec_revision_3m` (87 NaN): first 3 observations per ticker — no
#   3-month-ago value to compare against.
# These will be skipped during cross-sectional aggregation in the merge
# pipeline (cap-weighted mean across ~99 other stocks that month).
#
# **3 tickers with <24 months coverage (FBS, FLT, ONE):** same stocks as
# price targets — briefly in the top-100 universe then exited. Kept.
#
# **Factors retained: 13** (all kept)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 4: CLEAN & SAVE")
print("=" * 90)

factor_cols = [c for c in df.columns if c not in ['ticker', 'date']]

# ── 4a. Final NaN report ────────────────────────────────────────────────────
nan_check = df[factor_cols].isna().sum()
nan_cols = nan_check[nan_check > 0]
if len(nan_cols) == 0:
    print(f"\n  ✓ Zero NaN")
else:
    total_nan = nan_cols.sum()
    print(f"\n  Remaining NaN: {total_nan} (structural, left intentionally)")
    for col, n in nan_cols.items():
        print(f"    {col:<25s} {n:>5d} NaN")

# ── 4b. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Tickers: {df['ticker'].nunique()}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols)} columns):")
for i, c in enumerate(factor_cols, 1):
    vals = df[c].dropna()
    nan_n = df[c].isna().sum()
    nan_str = f"  ({nan_n} NaN)" if nan_n > 0 else ""
    print(f"    {i:>3d}. {c:<25s} range: [{vals.min():.4f}, {vals.max():.4f}]{nan_str}")

print(f"\n  Sample (first 5 rows):")
print(df.head(5).to_string(index=False))

# ── 4c. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'ibes_recommendations_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nCleaning complete.")

STAGE 4: CLEAN & SAVE

  Remaining NaN: 493 (structural, left intentionally)
    rec_dispersion              377 NaN
    rec_revision                 29 NaN
    rec_revision_3m              87 NaN

  Final shape: 49,110 rows × 15 columns
  Tickers: 230
  Date range: 2004-01-31 → 2024-12-31

  Factor list (13 columns):
      1. rec_mean                  range: [1.0000, 5.0000]
      2. rec_median                range: [1.0000, 5.0000]
      3. rec_numrec                range: [1.0000, 44.0000]
      4. rec_buy_pct               range: [0.0000, 100.0000]
      5. rec_sell_pct              range: [0.0000, 100.0000]
      6. rec_buy_sell_spread       range: [-100.0000, 100.0000]
      7. rec_dispersion            range: [0.0000, 2.1213]  (377 NaN)
      8. rec_revision              range: [-2.0000, 2.0000]  (29 NaN)
      9. rec_revision_3m           range: [-2.7500, 2.0000]  (87 NaN)
     10. rec_upgrades              range: [0.0000, 8.0000]
     11. rec_downgrades            range: [0.00